## Pre-requirements

In [ ]:
! pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu128 -U

In [ ]:
%pip install transformers==4.57.1 Pillow accelerate einops

### Optional requirement:
- flash-attention by Dao-AILab

You can set attn_implementation="sdpa" to avoid using/installing flash-attention.

For latest instructions on installation visit https://github.com/Dao-AILab/flash-attention and https://github.com/Dao-AILab/flash-attention/releases

The following cell provides useful information about your current environment that helps with choosing the correct artifact to download.

In [1]:
import sys
import platform
import torch

# Detect all the components
python_version = f"{sys.version_info.major}.{sys.version_info.minor}"
cp_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
torch_version = torch.__version__.split('+')[0]  # e.g., "2.8.0"
torch_major_minor = '.'.join(torch_version.split('.')[:2])  # e.g., "2.8"
cuda_version = torch.version.cuda  # e.g., "12.8"
cxx11_abi = torch.compiled_with_cxx11_abi()
arch = platform.machine()  # e.g., "x86_64" or "aarch64"

print("=" * 60)
print("Flash Attention Wheel Selector")
print("=" * 60)
print(f"  Python version:    {python_version} ({cp_tag})")
print(f"  PyTorch version:   {torch.__version__} (major.minor: {torch_major_minor})")
print(f"  CUDA version:      {cuda_version}")
print(f"  CXX11 ABI:         {cxx11_abi}")
print(f"  Architecture:      {arch}")
print(f"  Platform:          {sys.platform}")
print("=" * 60)

Flash Attention Wheel Selector
  Python version:    3.12 (cp312)
  PyTorch version:   2.8.0+cu128 (major.minor: 2.8)
  CUDA version:      12.8
  CXX11 ABI:         True
  Architecture:      x86_64
  Platform:          linux


For our case we need to download this specific package:

In [ ]:
! wget https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

In [ ]:
%pip install --no-dependencies --upgrade flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl

### Requirements for this notebook (but not for model):

In [ ]:
# dependencies for this notebook only
%pip install matplotlib

# Model instantiation

In [ ]:
# Auto-detect flash attention 2 availability
use_flash_attention = False
try:
    from flash_attn import flash_attn_func
    use_flash_attention = True
    import flash_attn
    print(f"✅ Flash Attention 2 detected (v{flash_attn.__version__})")
except (ImportError, RuntimeError, Exception) as e:
    print(f"⚠️  Flash Attention 2 not available: {e}")
    print("   Falling back to SDPA")

✅ Flash Attention 2 detected (v2.8.3)


In [ ]:
"""
Sample inference script for Phi-4-Vision-5B.
"""

import os
import torch
from transformers import AutoModelForCausalLM, AutoProcessor
from PIL import Image

device = "cuda:0"

model_path = "microsoft/Phi-4-reasoning-vision-15B"

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map=device,
    attn_implementation="flash_attention_2" if use_flash_attention else "sdpa",
    trust_remote_code=True,
)

processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

print(f"Model loaded on {model.device}")

In [ ]:
# Check the attention implementation
print(f"Attention implementation: {model.config._attn_implementation}")

### Helper functions

In [ ]:
## PLOT IMAGE WITH MODEL PREDICTION AND GROUND TRUTH

import ast

import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_boxes(image, model_output, correct_answer_norm):

    # Parse model output
    model_bbox = ast.literal_eval(model_output)  # [x1, y1, x2, y2] normalized

    # Convert normalized coords to absolute pixel coords
    img_w, img_h = image.size

    model_abs = [
        model_bbox[0] * img_w, model_bbox[1] * img_h,
        (model_bbox[2] - model_bbox[0]) * img_w, (model_bbox[3] - model_bbox[1]) * img_h
    ]  # x, y, w, h in pixels

    correct_abs = [
        correct_answer_norm[0] * img_w, correct_answer_norm[1] * img_h,
        (correct_answer_norm[2] - correct_answer_norm[0]) * img_w, (correct_answer_norm[3] - correct_answer_norm[1]) * img_h
    ]  # x, y, w, h in pixels

    fig, ax = plt.subplots(1, 1, figsize=(12, 7))
    ax.imshow(image)

    # Draw model prediction (red)
    rect_model = patches.Rectangle(
        (model_abs[0], model_abs[1]), model_abs[2], model_abs[3],
        linewidth=3, edgecolor='red', facecolor='none', linestyle='-', label='Model Prediction'
    )
    ax.add_patch(rect_model)

    # Draw correct answer (green)
    rect_correct = patches.Rectangle(
        (correct_abs[0], correct_abs[1]), correct_abs[2], correct_abs[3],
        linewidth=3, edgecolor='lime', facecolor='none', linestyle='--', label='Ground Truth'
    )
    ax.add_patch(rect_correct)

    ax.legend(fontsize=12, loc='upper right')
    ax.set_title(f"Model: {model_output}  |  GT: [{correct_answer_norm[0]:.3f}, {correct_answer_norm[1]:.3f}, {correct_answer_norm[2]:.3f}, {correct_answer_norm[3]:.3f}]", fontsize=13)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
def run_inference(processor, model, prompt, image, thinking_mode="hybrid"):
    ## FORM MESSAGE AND LOAD IMAGE
    messages = [
        {
            "role": "user", 
            "content": prompt,
        }
    ]

    ## PROCESS INPUTS 

    prompt = processor.tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        return_dict=False,
    )

    if thinking_mode == "think":
        prompt = str(prompt) + "<think>"
    elif thinking_mode == "nothink":
        prompt = str(prompt) + "<|dummy_84|>"

    print(f"Prompt: {prompt}")

    inputs = processor(text=prompt, images=[image], return_tensors="pt").to(model.device)

    ## GENERATE RESPONSE
    output_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        temperature=None,
        top_p=None,
        do_sample=False,
    )

    ## DECODE RESPONSE
    sequence_length = inputs["input_ids"].shape[1]

    sequence_length -= 1 if thinking_mode == "think" else 0 # remove the extra token for nothink mode

    new_output_ids = output_ids[:, sequence_length:]
    model_output = processor.batch_decode(
        new_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    return model_output

## Grounding Example

In [ ]:
### GROUNDING EXAMPLE

prompt = "<image>\nLocate the the UI element this instruction describes: start playback. Output its bbox coordinates using the format of [x1, y1, x2, y2] using relative coordinates from 0.0000 to 1.0000."
image_file = "pc_0c634288-11b9-498f-8fb6-582e26ea17c9.png"

image_path = os.path.join(".", image_file)
print(f"Loading image from {image_path}")
image = Image.open(image_path).convert("RGB")

model_output = run_inference(processor, model, prompt, image, thinking_mode="hybrid")

correct_answer = [165,467,53,52] # in xywh format, absolute coordinates

print(f'Response: {model_output}')
print("Expected Answer: [0.174, 0.868, 0.224, 0.977]")

correct_answer_norm = [
    correct_answer[0]/image.size[0],
    correct_answer[1]/image.size[1],
    (correct_answer[0]+correct_answer[2])/image.size[0],
    (correct_answer[1]+correct_answer[3])/image.size[1],
] # convert to normalized xyxy format

print(f"Correct Answer: {correct_answer_norm}")

plot_boxes(image, model_output, correct_answer_norm)

## Image description example:

In [ ]:
### DESCRIPTION EXAMPLE
prompt = "<image>\nDescribe this image in detail."
image_file = "000a045a0715d64d.jpg"

### GROUNDING EXAMPLE
image_path = os.path.join(".", image_file)
print(f"Loading image from {image_path}")
image = Image.open(image_path).convert("RGB")

model_output = run_inference(processor, model, prompt, image, thinking_mode="hybrid")

print(f'Response: {model_output}')
